In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

Libraries loaded ✓


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

Libraries loaded ✓


In [3]:
DATA_PATH = 'itineraries.csv'

ROUTES = [
    ('PDX', 'LAX'),
    ('PDX', 'JFK'),
    ('PDX', 'ORD'),
    ('PDX', 'SFO'),
    ('PDX', 'SEA'),
]

COLS = [
    'searchDate', 'flightDate',
    'startingAirport', 'destinationAirport',
    'isNonStop', 'isBasicEconomy',
    'baseFare', 'totalFare',
    'seatsRemaining', 'travelDuration',
    'segmentsAirlineName', 'segmentsCabinCode',
]

origins = {r[0] for r in ROUTES}
dests   = {r[1] for r in ROUTES}

chunks = []
chunk_size = 100_000

print('Loading data in chunks...')
for i, chunk in enumerate(pd.read_csv(DATA_PATH, usecols=COLS,
                                       chunksize=chunk_size, low_memory=False)):
    filtered = chunk[
        chunk['startingAirport'].isin(origins) &
        chunk['destinationAirport'].isin(dests)
    ]
    if len(filtered) > 0:
        chunks.append(filtered)
    if i % 100 == 0:
        print(f'  processed {i * chunk_size:,} rows...')

df = pd.concat(chunks, ignore_index=True)
print(f'\nDone! Filtered rows: {len(df):,}')

Loading data in chunks...
  processed 0 rows...
  processed 10,000,000 rows...
  processed 20,000,000 rows...
  processed 30,000,000 rows...
  processed 40,000,000 rows...
  processed 50,000,000 rows...
  processed 60,000,000 rows...
  processed 70,000,000 rows...
  processed 80,000,000 rows...


ValueError: No objects to concatenate

In [4]:
temp = pd.read_csv(DATA_PATH, nrows=5)
print(temp.columns.tolist())

['legId', 'searchDate', 'flightDate', 'startingAirport', 'destinationAirport', 'fareBasisCode', 'travelDuration', 'elapsedDays', 'isBasicEconomy', 'isRefundable', 'isNonStop', 'baseFare', 'totalFare', 'seatsRemaining', 'totalTravelDistance', 'segmentsDepartureTimeEpochSeconds', 'segmentsDepartureTimeRaw', 'segmentsArrivalTimeEpochSeconds', 'segmentsArrivalTimeRaw', 'segmentsArrivalAirportCode', 'segmentsDepartureAirportCode', 'segmentsAirlineName', 'segmentsAirlineCode', 'segmentsEquipmentDescription', 'segmentsDurationInSeconds', 'segmentsDistance', 'segmentsCabinCode']


In [5]:
temp = pd.read_csv(DATA_PATH, usecols=['startingAirport'], nrows=500_000)
print(temp['startingAirport'].value_counts().head(20))

startingAirport
LAX    49121
LGA    38606
BOS    36726
DFW    34380
ORD    34205
SFO    34076
ATL    32806
CLT    32567
MIA    32005
PHL    29254
DTW    28495
DEN    28191
EWR    25686
JFK    23619
IAD    20636
OAK    19627
Name: count, dtype: int64


In [6]:
temp2 = pd.read_csv(DATA_PATH, usecols=['startingAirport', 'destinationAirport'], nrows=500_000)
print('Origins:', temp2['startingAirport'].unique().tolist())
print('Destinations:', temp2['destinationAirport'].unique().tolist())

Origins: ['ATL', 'BOS', 'CLT', 'DEN', 'DFW', 'DTW', 'EWR', 'IAD', 'JFK', 'LAX', 'LGA', 'MIA', 'OAK', 'ORD', 'PHL', 'SFO']
Destinations: ['BOS', 'CLT', 'DEN', 'DFW', 'DTW', 'EWR', 'IAD', 'JFK', 'LAX', 'LGA', 'MIA', 'OAK', 'ORD', 'PHL', 'SFO', 'ATL']


In [7]:
DATA_PATH = 'itineraries.csv'

ROUTES = [
    ('LAX', 'JFK'),  # West Coast → East Coast
    ('LAX', 'ORD'),  # West Coast → Midwest
    ('SFO', 'JFK'),  # Bay Area → NYC
    ('BOS', 'LAX'),  # East Coast → West Coast
    ('ATL', 'LAX'),  # South → West Coast
]

COLS = [
    'searchDate', 'flightDate',
    'startingAirport', 'destinationAirport',
    'isNonStop', 'isBasicEconomy',
    'baseFare', 'totalFare',
    'seatsRemaining', 'travelDuration',
    'segmentsAirlineName', 'segmentsCabinCode',
]

origins = {r[0] for r in ROUTES}
dests   = {r[1] for r in ROUTES}

chunks = []
chunk_size = 100_000

print('Loading data in chunks...')
for i, chunk in enumerate(pd.read_csv(DATA_PATH, usecols=COLS,
                                       chunksize=chunk_size, low_memory=False)):
    filtered = chunk[
        chunk['startingAirport'].isin(origins) &
        chunk['destinationAirport'].isin(dests)
    ]
    if len(filtered) > 0:
        chunks.append(filtered)
    if i % 100 == 0:
        print(f'  processed {i * chunk_size:,} rows...')

df = pd.concat(chunks, ignore_index=True)
print(f'\nDone! Filtered rows: {len(df):,}')

Loading data in chunks...
  processed 0 rows...
  processed 10,000,000 rows...
  processed 20,000,000 rows...
  processed 30,000,000 rows...
  processed 40,000,000 rows...
  processed 50,000,000 rows...
  processed 60,000,000 rows...
  processed 70,000,000 rows...
  processed 80,000,000 rows...

Done! Filtered rows: 5,129,145


In [8]:
# Parse dates
df['searchDate'] = pd.to_datetime(df['searchDate'])
df['flightDate'] = pd.to_datetime(df['flightDate'])

# Days between search and flight (booking window)
df['days_out'] = (df['flightDate'] - df['searchDate']).dt.days

# Calendar features from flightDate
df['flight_month']    = df['flightDate'].dt.month
df['flight_dow']      = df['flightDate'].dt.dayofweek   # 0=Mon, 6=Sun
df['flight_dow_name'] = df['flightDate'].dt.day_name()

# Route string for grouping
df['route'] = df['startingAirport'] + '_' + df['destinationAirport']

# Parse duration to minutes
def parse_duration(s):
    """Convert 'PT2H35M' to minutes."""
    try:
        s = str(s).replace('PT', '')
        h = int(s.split('H')[0]) if 'H' in s else 0
        m = int(s.split('H')[-1].replace('M', '')) if 'M' in s else 0
        return h * 60 + m
    except:
        return np.nan

df['duration_min'] = df['travelDuration'].apply(parse_duration)

# Drop rows with missing fare or negative days_out
df = df.dropna(subset=['baseFare', 'days_out'])
df = df[df['days_out'] >= 0]

print(f'Clean rows: {len(df):,}')
print(df[['route', 'days_out', 'flight_month', 'baseFare']].describe())

Clean rows: 5,129,145
           days_out  flight_month      baseFare
count  5.129145e+06  5.129145e+06  5.129145e+06
mean   2.711836e+01  7.814005e+00  2.842218e+02
std    1.628927e+01  1.639774e+00  1.897413e+02
min    1.000000e+00  4.000000e+00  4.100000e-01
25%    1.300000e+01  7.000000e+00  1.553500e+02
50%    2.600000e+01  8.000000e+00  2.595300e+02
75%    4.000000e+01  9.000000e+00  3.795300e+02
max    6.000000e+01  1.100000e+01  4.566510e+03
